# Web Scraping Workshop: Scraping Books to Scrape

Welcome to the live web scraping lab! In this workshop, we will scrape data from a real online bookstore sandbox: [Books to Scrape](http://books.toscrape.com/).

This notebook is divided into two distinct parts:
1. **Part 1 — Guided Exercises (With Your Instructor):** Learn how to fetch live web pages with `requests`, check HTTP status codes, inspect HTML attributes, clean currency prices, and build full links.
2. **Part 2 — Solo Practice (For Students):** Practice extracting star ratings, checking stock status, creating a complete page scraper, and analyzing the data with `pandas`.

---

## Learning Objectives

By the end of this workshop, you will be able to:
- Send HTTP GET requests using Python's `requests` library and handle response status codes (`200 OK`)
- Use BeautifulSoup to parse live HTML document trees
- Extract data hidden inside HTML tag attributes (such as `title="..."` and `href="..."`)
- Convert multi-class CSS tags (such as `star-rating Three`) into numeric values
- Aggregate scraped data into a structured list of dictionaries
- Filter, summarize, and analyze scraped web data using `pandas`

**Goal:** Build a production-grade web scraper extracting book titles, prices, ratings, availability, and URLs into a clean Pandas DataFrame.

In [ ]:
# Setup: Environment check, imports, and offline zero-404 fallback check
import sys
import os
import re
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Environment check
print(f"Python version: {sys.version.split()[0]} | Platform: {sys.platform}")

# 2. Network connectivity check with offline cache fallback (Zero-404 Rule)
BASE_URL = "http://books.toscrape.com/"
CACHE_FILE = "books_toscrape_cache.html"

def load_sandbox_html():
    """Fetches the live website or loads from local cache if offline."""
    try:
        resp = requests.get(BASE_URL, timeout=5)
        if resp.status_code == 200:
            resp.encoding = "utf-8"
            # Cache locally for offline student work
            with open(CACHE_FILE, "w", encoding="utf-8") as f:
                f.write(resp.text)
            print("🌐 Live network connected: Successfully fetched books.toscrape.com!")
            return resp.text
    except Exception as err:
        print(f"⚠️ Network notice ({err}). Switching to offline local cache.")
    
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            print("📁 Loaded books from local offline cache.")
            return f.read()
    else:
        raise RuntimeError("No internet and no cache file found. Please ensure books_toscrape_cache.html is present.")

# Warm up the sandbox data
RAW_HTML_DATA = load_sandbox_html()
print(f"HTML ready: {len(RAW_HTML_DATA):,} characters loaded.")
print("✅ Setup complete: requests, BeautifulSoup, and pandas are ready!")

## Part 1 — Guided Exercises (With Your Instructor)

### 1.1 Sending an HTTP GET Request & Checking Status Code

To scrape a live website, our Python script acts like a web browser: it sends an **HTTP GET request** across the internet to the web server, which replies with an **HTTP Response**.

```text
Python (requests.get)  ─── HTTP GET Request ───►  books.toscrape.com
Python (response)      ◄─── HTTP 200 OK (HTML) ──  Web Server
```

```text
response = requests.get(url, timeout=10)
           │        │   │   └─ safety net: max seconds to wait before giving up
           │        │   └─ target web URL
           │        └─ sends HTTP GET request
           └─ returns a Response object containing status_code and text
```

#### The HTTP Status Code Rule
Before doing anything with a web response, **always check the status code**:
- `200` = **OK (Success!)** The page was found and returned.
- `404` = **Not Found.** The URL does not exist.
- `500` = **Internal Server Error.** The website crashed.

**Expected output:** `Status Code: 200` and HTML character count.

In [ ]:
# 1.1 — Fetch the homepage of books.toscrape.com
url = "http://books.toscrape.com/"

# TODO 1: Use requests.get() to request the URL with a 10-second timeout
response = requests.get(..., timeout=10)
response.encoding = "utf-8"  # Ensure proper encoding for currency symbols

print(f"Status Code: {response.status_code}")
print(f"Downloaded HTML size: {len(response.text):,} characters")

# Self-check assertions
assert response.status_code == 200, f"Expected status code 200, got {response.status_code}"
assert len(response.text) > 0, "Response body should not be empty!"
print("✅ Test passed!")

### 1.2 Parsing HTML & Finding Book Containers

Now that we have the raw HTML in `response.text`, we pass it into `BeautifulSoup`.

Inspecting the structure of `books.toscrape.com`, every single book item is packaged inside an `<article>` tag with the class `"product_pod"`:

```text
<article class="product_pod">                                ← Book Container
 ├── <div class="image_container"> ... </div>
 ├── <p class="star-rating Three"> ... </p>                  ← Star Rating
 ├── <h3>
 │    └── <a href="..." title="Full Title">Truncated...</a>   ← Title & Link
 └── <div class="product_price">
      ├── <p class="price_color">£51.77</p>                  ← Price
      └── <p class="instock availability">In stock</p>        ← Availability
```

Let's parse the document and find all 20 book pods on the page.

**Expected output:** `Found 20 books on page 1.`

In [ ]:
# 1.2 — Parse the HTML with BeautifulSoup and locate all book articles
soup = BeautifulSoup(response.text, "html.parser")

# TODO 1: Find all <article> tags with class_ "product_pod"
all_books = soup.find_all("article", class_=...)

print(f"Found {len(all_books)} books on page 1.")

# Self-check assertion
assert len(all_books) == 20, f"Expected 20 books on page 1, got {len(all_books)}"
print("✅ Test passed!")

### 1.3 Extracting Full Book Titles: Text vs. Attribute

Let's inspect how the title of the first book is stored inside its `<h3>` tag:

```html
<h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
    A Light in the ...
  </a>
</h3>
```

Look closely at the differences:
1. `tag.text` is `"A Light in the ..."` — **it is truncated with an ellipsis!**
2. The HTML attribute `title="..."` contains the **complete, untruncated title**: `"A Light in the Attic"`.

In BeautifulSoup, you access HTML attributes just like keys in a Python dictionary:
```text
complete_title = book.h3.a["title"]
                 │    │  │ └─ accesses the 'title' attribute value inside <a ...>
                 │    │  └─ the <a> tag
                 │    └─ the <h3> tag inside book
                 └─ the <article class="product_pod"> tag
```

**Expected output:** Visible text `'A Light in the ...'` and Full title `'A Light in the Attic'`.

In [ ]:
# 1.3 — Compare visible text vs the 'title' attribute on the first book
first_book = all_books[0]

visible_text = first_book.h3.a.text.strip()

# TODO 1: Extract the 'title' attribute from first_book.h3.a using dictionary syntax ['title']
full_title = first_book.h3.a[...]

print(f"Visible (truncated) text: '{visible_text}'")
print(f"Full (attribute) title:   '{full_title}'")

# Self-check assertion
assert full_title == "A Light in the Attic", f"Expected 'A Light in the Attic', got '{full_title}'"
print("✅ Test passed!")

### 1.4 Extracting and Cleaning Book Prices

The price of each book lives in a `<p>` tag with class `"price_color"`.
When we extract its text, we get a string like `"£51.77"`.

Because Python cannot perform arithmetic on currency symbols, we clean the string using a simple regular expression `re.sub(r"[^\d.]", "", raw_price)` or by stripping the prefix:

```text
clean_price = float(re.sub(r"[^\d.]", "", raw_price))
              │     │      │          │  └─ original string '£51.77'
              │     │      │          └─ replace with empty string
              │     │      └─ pattern: any character that is NOT a digit or a dot
              └─ convert the clean string '51.77' to float 51.77
```

Let's extract and clean the price of the first book.

**Expected output:** `Cleaned price: 51.77 (type: float)`

In [ ]:
# 1.4 — Extract and clean the price of the first book
price_tag = first_book.find("p", class_="price_color")
raw_price = price_tag.text.strip()

# TODO 1: Remove any non-numeric characters and convert to float
# Hint: Use re.sub(r"[^\d.]", "", raw_price) and float(...)
clean_price = float(...)

print(f"Raw price text: '{raw_price}'")
print(f"Cleaned price:   {clean_price} (type: {type(clean_price).__name__})")

# Self-check assertion
assert clean_price == 51.77, f"Expected 51.77, got {clean_price}"
print("✅ Test passed!")

### 1.5 Resolving Relative URLs to Absolute Links

Webpages frequently use **relative links** inside `href` attributes:
`"catalogue/a-light-in-the-attic_1000/index.html"`

If you try to open this relative link directly, your computer won't know which website it belongs to! We need the **absolute URL**:
`"http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"`

Python's built-in `urljoin` function safely merges a base URL with any relative path:

```text
from urllib.parse import urljoin

full_url = urljoin("http://books.toscrape.com/", relative_link)
           │       │                             └─ e.g. "catalogue/book_1/index.html"
           │       └─ base homepage URL
           └─ cleanly resolves relative links into complete, clickable URLs
```

Let's extract the `href` attribute from the first book and resolve its full URL.

**Expected output:** `Full book URL: http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html`

In [ ]:
# 1.5 — Extract the relative href and build the full URL
# TODO 1: Extract the 'href' attribute from first_book.h3.a
relative_link = first_book.h3.a[...]

# TODO 2: Use urljoin(url, relative_link) to build the full link
full_url = urljoin(url, ...)

print(f"Relative link: {relative_link}")
print(f"Full book URL: {full_url}")

# Self-check assertions
assert full_url == "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html", f"Unexpected URL: {full_url}"
print("✅ Test passed!")

## Part 2 — Solo Practice (For Students)

Now it is your turn to apply what you've learned! Work through these solo challenges to extract remaining features and build a complete book scraper.

---

### 2.1 Solo Task: Extracting Star Ratings

Inspect the rating element on the first book:
```html
<p class="star-rating Three">
  <i class="icon-star"></i> ...
</p>
```

Notice that there is no written rating text inside the tag! Instead, the rating is stored in the **CSS class name**:
- One star: `class="star-rating One"`
- Three stars: `class="star-rating Three"`
- Five stars: `class="star-rating Five"`

When you access `tag["class"]` in BeautifulSoup, it returns a **list of class strings**:
`['star-rating', 'Three']`

The rating word is at index 1 (`classes[1]`). You can convert this word into a number using a dictionary:
```python
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
```

**Task:** Extract the star rating for `first_book` as an integer (`1` to `5`).

**Expected output:** `Rating word: 'Three' -> Numeric rating: 3 stars`

In [ ]:
# 2.1 — Solo Task: Extract the star rating as an integer (1 to 5)
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# TODO 1: Find the <p> tag with class_ "star-rating" inside first_book
rating_tag = first_book.find("p", class_=...)

# TODO 2: Extract the list of classes using rating_tag["class"]
classes = ...

# TODO 3: Get the second class name (index 1 of classes)
rating_word = ...

# TODO 4: Look up rating_word in RATING_MAP to get the integer rating
rating_number = RATING_MAP.get(..., 0)

print(f"Classes list: {classes}")
print(f"Rating word: '{rating_word}' -> Numeric rating: {rating_number} stars")

# Self-check assertion
assert rating_number == 3, f"Expected 3 stars, got {rating_number}"
print("✅ Test passed!")

### 2.2 Solo Task: Checking Stock Availability

Look at the availability element on the first book:
```html
<p class="instock availability">
    <i class="icon-ok"></i>
    In stock
</p>
```

When you extract `.text.strip()`, you get the string `"In stock"`.

**Task:**
1. Locate the `<p>` tag with class `"instock"` inside `first_book`.
2. Extract the cleaned text using `.text.strip()`.
3. Check if `"in stock"` is in the lowercase text to create a boolean `is_in_stock` (`True` or `False`).

**Expected output:** `Cleaned stock text: 'In stock'`, `Is in stock (boolean): True`

In [ ]:
# 2.2 — Solo Task: Check if the first book is in stock
# TODO 1: Find the <p> tag with class_ "instock" in first_book
stock_tag = first_book.find("p", class_=...)

# TODO 2: Extract and strip the text from stock_tag
stock_text = ...

# TODO 3: Check if "in stock" appears in stock_text.lower() (True or False)
is_in_stock = ...

print(f"Cleaned stock text: '{stock_text}'")
print(f"Is in stock (boolean): {is_in_stock}")

# Self-check assertion
assert is_in_stock is True, "First book should be in stock!"
print("✅ Test passed!")

### 2.3 Solo Task: Building the Full Page Scraper

Now assemble everything you have practiced into a loop that scrapes all 20 books on page 1!

**Task:**
Iterate over `all_books`. For each book, extract:
- `title`: untruncated string from `book.h3.a["title"]`
- `price`: cleaned `float` from `<p class="price_color">`
- `rating`: integer (`1` to `5`) from `<p class="star-rating">`
- `in_stock`: boolean (`True`/`False`) from `<p class="instock">`
- `url`: full absolute URL resolved with `urljoin(url, book.h3.a["href"])`

Store each book as a dictionary inside a list named `catalog`.

**Expected output:** `Successfully scraped 20 books!` with book 1 having title `'A Light in the Attic'` and price `51.77`.

In [ ]:
# 2.3 — Solo Task: Build the complete scraping loop for all 20 books
catalog = []

for book in all_books:
    # TODO 1: Extract full title from book.h3.a["title"]
    b_title = ...
    
    # TODO 2: Extract and clean the price float
    raw_p = book.find("p", class_="price_color").text
    b_price = ...
    
    # TODO 3: Extract the rating number (1-5) using RATING_MAP
    b_classes = book.find("p", class_="star-rating")["class"]
    b_rating_word = b_classes[1] if len(b_classes) > 1 else "Unknown"
    b_rating = ...
    
    # TODO 4: Extract boolean in-stock status
    b_stock_text = book.find("p", class_="instock").text.strip()
    b_in_stock = ...
    
    # TODO 5: Build full absolute URL using urljoin(url, book.h3.a["href"])
    b_url = ...
    
    # TODO 6: Append dictionary to catalog
    catalog.append({
        "title": b_title,
        "price": b_price,
        "rating": b_rating,
        "in_stock": b_in_stock,
        "url": b_url
    })

print(f"Successfully scraped {len(catalog)} books!")
print()
print("First book record:")
print(catalog[0])

# Self-check assertions
assert len(catalog) == 20, f"Expected 20 books, got {len(catalog)}"
assert catalog[0]["price"] == 51.77, f"Expected 51.77, got {catalog[0]['price']}"
assert catalog[0]["rating"] == 3, f"Expected 3 stars, got {catalog[0]['rating']}"
assert catalog[0]["in_stock"] is True, "First book should be in stock"
print("✅ Test passed!")

### 2.4 Solo Task: Data Analysis & Filtering with Pandas

Now that your data is cleanly structured, load `catalog` into a **Pandas DataFrame** to perform quick data analysis:

```python
df = pd.DataFrame(catalog)
```

**Tasks:**
1. Filter all books with `price < 20.0` into `cheap_books`.
2. Filter all books with `rating == 5` into `top_rated_books`.
3. Calculate the overall average price of all 20 books, rounded to 2 decimal places (`avg_price`).

**Expected output:** Books under £20: `3`, 5-star books: `4`, Average price: `£38.05`.

In [ ]:
# 2.4 — Solo Task: Analyze the scraped books using Pandas
df = pd.DataFrame(catalog)

# TODO 1: Filter books with price < 20.0 into cheap_books
cheap_books = ...

# TODO 2: Filter books with rating == 5 into top_rated_books
top_rated_books = ...

# TODO 3: Calculate the mean price of df["price"] rounded to 2 decimal places
avg_price = round(..., 2)

print(f"Total books analyzed: {len(df)}")
print(f"Books under £20:      {len(cheap_books)}")
print(f"5-star rated books:   {len(top_rated_books)}")
print(f"Average book price:   £{avg_price:.2f}")

print()
print("Affordable books (< £20):")
print(cheap_books[["title", "price", "rating"]].to_string(index=False))

# Self-check assertions
assert len(cheap_books) == 3, f"Expected 3 books under £20, got {len(cheap_books)}"
assert len(top_rated_books) == 4, f"Expected 4 top-rated books, got {len(top_rated_books)}"
assert avg_price == 38.05, f"Expected 38.05 average price, got {avg_price}"
print("✅ Test passed!")

## Bug Hunt — read the traceback like a detective

The cell below contains a very common attribute lookup bug: `KeyError: 'title'`.

Run the cell first and let it fail. Notice the traceback:

```text
Traceback (most recent call last):
  File "<ipython-input>", line 4, in <module>
    title = first_book.h3["title"]
KeyError: 'title'
│         └─ Python says: <h3> has no HTML attribute named 'title'!
└─ the error type
```

### Why did this happen?
The student assumed that the `title` attribute is located on the `<h3>` header tag. But if you inspect the HTML:
- `<h3>` is just an outer wrapper.
- The `title` attribute is on the `<a>` tag *inside* `<h3>`: `<h3><a title="...">...</a></h3>`.

Inspect the hierarchy, fix the tag navigation, and re-run the cell.

**Goal:** Fix the attribute lookup so the book title prints cleanly.

In [ ]:
# Bug Hunt — run this cell, read the traceback, then fix the attribute access

# A student tried to extract the book title, but targeted the wrong tag:
title = first_book.h3["title"]  # BUG: <h3> does not have a 'title' attribute!

print("Book title:", title)

## Reflection

Answer these questions in your own words:

1. Why does `books.toscrape.com` store truncated text like `"A Light in the ..."` in `.text`, but provide the complete title inside the HTML attribute `title="A Light in the Attic"`? When scraping, how do you decide whether to look at `.text` or an attribute?

   **Answer:**
   
   __________________________________________________________________________________________

2. When you access `tag["class"]` in BeautifulSoup, why does it return a list of strings (such as `['star-rating', 'Three']`), whereas `tag["href"]` returns a single string?

   **Answer:**
   
   __________________________________________________________________________________________

3. Why is it essential to check `response.status_code == 200` before parsing HTML when scraping live web pages? What could happen to your scraper if you skip this check?

   **Answer:**
   
   __________________________________________________________________________________________

**Goal:** Explain text vs. attribute decisions, multi-valued class attributes, and defensive HTTP status checking.